<a href="https://colab.research.google.com/github/paulheather147/FinalYearProject/blob/main/FinalModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# imports for dataset, tensorflow, keras, and other utilities

!pip install -q datasets

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, EfficientNetB2
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from datasets import load_dataset, Dataset

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import numpy as np

from PIL import Image
import os
import matplotlib.pyplot as plt
import cv2
import torch
import glob
import math

from keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, multiply, Permute, Concatenate, Conv2D, Add, Activation, Lambda
from tensorflow.keras import backend as K

!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-2ee49f9c-dc12-f140-67be-d31ef7e06e86)


In [ ]:
# save package requirements to use in my app
!pip freeze --local > colab_requirements.txt

In [2]:
# loads the Alzheimer_MRI dataset from Hugging Face
dataset = load_dataset("Falah/Alzheimer_MRI")
print(dataset)
print(dataset["train"].features)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-c08a401c53fe53(…):   0%|          | 0.00/22.6M [00:00<?, ?B/s]

data/test-00000-of-00001-44110b9df98c558(…):   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1280 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5120
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1280
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented'])}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# splits dataset into training, validation and test splits
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train = train_val_split["train"]
val = train_val_split["test"]
test = dataset["test"]

# load synthetic Moderate_Demented images generated using StyleGAN2-ADA
synth_dir = "/content/drive/MyDrive/Moderate_Demented_Synthetic"
synth_files = sorted(glob.glob(f"{synth_dir}/seed*.png"))
print("Found", len(synth_files), "synthetic images.")

def load_synth_example(path):
    # load each synthetic image as greyscale and assign label 1
    img = Image.open(path).convert("L")
    img = np.array(img)
    return {"image": img, "label": 1}

synth_data = [load_synth_example(p) for p in synth_files]
synth_dataset = Dataset.from_list(synth_data)
print(synth_dataset)

# ensure greyscale images always have a chennel dimension
def ensure_channel_dim(images):
    current_rank = tf.rank(images)

    def add_channel():
        return tf.expand_dims(images, axis=-1)

    def keep_same():
        return images

    return tf.cond(tf.equal(current_rank, 2), add_channel, keep_same)

# ensures images are in RBG format
def ensure_rgb_channels(images):
    current_channels = tf.shape(images)[-1]

    def to_rgb():
        return tf.image.grayscale_to_rgb(images)

    def drop_alpha():
        return images[..., :3]

    def keep_same():
        return images

    images = tf.cond(tf.equal(current_channels, 1), to_rgb, keep_same)

    new_channels = tf.shape(images)[-1]

    def keep_images_rgb():
        return images

    images = tf.cond(tf.equal(new_channels, 4), drop_alpha, keep_images_rgb)
    return images

import tensorflow as tf

# define four augmentations once
rotation_layer   = tf.keras.layers.RandomRotation(0.03)
zoom_layer       = tf.keras.layers.RandomZoom(0.05)
contrast_layer   = tf.keras.layers.RandomContrast(0.1)
brightness_layer = tf.keras.layers.RandomBrightness(0.1)

# applies exactly one random augmentation per image
def random_single_augmentation(image):
    choice = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)

    def rotation_fn(): return rotation_layer(image, training=True)
    def zoom_fn(): return zoom_layer(image, training=True)
    def contrast_fn(): return contrast_layer(image, training=True)
    def brightness_fn(): return brightness_layer(image, training=True)

    return tf.switch_case(choice, branch_fns=[rotation_fn, zoom_fn, contrast_fn, brightness_fn])

# convert HF dataset splits to tf.data.Dataset objects
def to_tensorflow_dataset(dataset_split, image_size, preprocess_fn, augment=False, shuffle=False):
        dataset_tf = dataset_split.to_tf_dataset(
        columns=["image", "label"],
        shuffle=shuffle,
        num_workers=0,
    )
        def preprocess_input(example_dict):
            images = tf.cast(example_dict["image"], tf.float32)
            images = ensure_channel_dim(images)
            images = ensure_rgb_channels(images)
            images.set_shape([None, None, 3])
            images = tf.image.resize(images, (image_size, image_size))

            if augment:
                images = random_single_augmentation(images)

            images = preprocess_fn(images)

            labels = tf.cast(example_dict["label"], tf.int32)
            labels = tf.one_hot(labels, depth=4)
            return images, labels

        dataset_tf = dataset_tf.map(preprocess_input,
                                    num_parallel_calls=tf.data.AUTOTUNE)
        dataset_tf = dataset_tf.batch(50)
        return dataset_tf.prefetch(tf.data.AUTOTUNE)

# build training datasets for vgg and efficientnet
vgg_clean_train = to_tensorflow_dataset(train, 224, vgg_preprocess, augment=False, shuffle=True)
eff_clean_train = to_tensorflow_dataset(train, 260, eff_preprocess, augment=False, shuffle=True)

# synthetic only datasets
vgg_synth_train = to_tensorflow_dataset(synth_dataset, 224, vgg_preprocess, augment=False, shuffle=True)
eff_synth_train = to_tensorflow_dataset(synth_dataset, 260, eff_preprocess, augment=False, shuffle=True)

# real clean images combined with synthetic images
full_vgg_synth_train = vgg_synth_train.concatenate(vgg_clean_train)
full_eff_synth_train = eff_synth_train.concatenate(eff_clean_train)

# augmented training dataset, and unaugmented validation and test splits
vgg_train_dataset = to_tensorflow_dataset(train, 224, vgg_preprocess, augment=True, shuffle=True)
vgg_val_dataset = to_tensorflow_dataset(val, 224, vgg_preprocess, augment=False, shuffle=False)
vgg_test_dataset = to_tensorflow_dataset(test, 224, vgg_preprocess, augment=False, shuffle=False)

eff_train_dataset = to_tensorflow_dataset(train, 260, eff_preprocess, augment=True, shuffle=True)
eff_val_dataset = to_tensorflow_dataset(val, 260, eff_preprocess, augment=False, shuffle=False)
eff_test_dataset = to_tensorflow_dataset(test, 260, eff_preprocess, augment=False, shuffle=False)

vgg_full_train_dataset = vgg_train_dataset
eff_full_train_dataset = eff_train_dataset


Found 50 synthetic images.
Dataset({
    features: ['image', 'label'],
    num_rows: 50
})


In [4]:
# CBAM (convolutional block attention module)
#channel attention
def channel_attention(x, ratio=8):
    ch = K.int_shape(x)[-1]
    hidden = max(ch // ratio, 1)

    shared_1 = layers.Dense(hidden, activation="relu", kernel_initializer="he_normal", use_bias=True)
    shared_2 = layers.Dense(ch, kernel_initializer="he_normal", use_bias=True)

    avg = layers.GlobalAveragePooling2D(keepdims=True)(x)
    mx  = layers.GlobalMaxPooling2D(keepdims=True)(x)

    attn = layers.Add()([shared_2(shared_1(avg)), shared_2(shared_1(mx))])
    attn = layers.Activation("sigmoid")(attn)
    return layers.Multiply()([x, attn])

# spatial attention
def spatial_attention(x, kernel_size=7):
    avg = tf.keras.ops.mean(x, axis=-1, keepdims=True)
    mx  = tf.keras.ops.max(x, axis=-1, keepdims=True)
    concat = layers.Concatenate(axis=-1)([avg, mx])
    attn = layers.Conv2D(
        1, kernel_size, padding="same", activation="sigmoid",
        kernel_initializer="he_normal", use_bias=False
    )(concat)
    return layers.Multiply()([x, attn])

# full cbam block
def cbam_block(x, ratio=8):
    y = channel_attention(x, ratio)
    y = spatial_attention(y)
    return layers.Add()([x, y])

In [5]:
# VGG backbone with ImageNet pretrained weights
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)

#Freeze backbone for initial training
vgg_base.trainable = False

# VGG16 classifier with CBAM included
vgg_inputs = tf.keras.Input(shape=(224, 224, 3))
x = vgg_base(vgg_inputs, training=False)
x = cbam_block(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
vgg_outputs = layers.Dense(4, activation="softmax")(x)

vgg_model = tf.keras.Model(vgg_inputs, vgg_outputs)

vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

vgg_model.summary()

# use add_1 for gradcam

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 7, 7, 512) │ 14,714,688 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1, 1, 512) │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 1, 1, 512) │          0 │ vgg16[0][0]       │
│ (GlobalMaxPooling2… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1, 1, 64)  │     32,832 │ global_average_p… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1, 1, 512) │     33,280 │ dense[0][0],      │
│                     │                   │            │ dense[1][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1, 1, 512) │          0 │ dense_1[0][0],    │
│                     │                   │            │ dense_1[1][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 1, 1, 512) │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 7, 7, 512) │          0 │ vgg16[0][0],      │
│                     │                   │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mean (Mean)         │ (None, 7, 7, 1)   │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max (Max)           │ (None, 7, 7, 1)   │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 7, 7, 2)   │          0 │ mean[0][0],       │
│ (Concatenate)       │                   │            │ max[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 7, 7, 1)   │         98 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 7, 7, 512) │          0 │ multiply[0][0],   │
│ (Multiply)          │                   │            │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 7, 7, 512) │          0 │ vgg16[0][0],      │
│                     │                   │            │ multiply_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ add_1[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    131,328 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 4)         │      1,028 │ dropout[0][0]   

 Total params: 14,913,254 (56.89 MB)

 Trainable params: 198,566 (775.65 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [6]:
VGG_EPOCHS = 50

# class weights to help with class imbalance
class_weight = {
    0: 1.25,
    1: 2.0,
    2: 1.0,
    3: 1.0,
}

# reduce learning rate when validation accuracy plateaus
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

# stop early and restore best weights when validation accuracy doesn't increase over the space of 12 epochs
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

# train for 1 epoch on clean original data
vgg_history_phase0 = vgg_model.fit(
    vgg_clean_train,
    validation_data=vgg_val_dataset,
    epochs=1,
    class_weight=class_weight,
)

# train briefly on clean & synthetic data
vgg_history_phase1 = vgg_model.fit(
    full_vgg_synth_train,
    validation_data=vgg_val_dataset,
    epochs=10,
    class_weight=class_weight,
    callbacks=[reduce_lr, early_stop]
)

# train on augmented data
vgg_history_phase2 = vgg_model.fit(
    vgg_full_train_dataset,
    validation_data = vgg_val_dataset,
    epochs = VGG_EPOCHS,
    class_weight = class_weight,
    callbacks = [reduce_lr, early_stop]
)

# unfreeze top layers of VGG16 to allow for fine tuning
print("\n=== PHASE 2: Fine-tuning top layers ===")
vgg_base.trainable = True
for layer in vgg_base.layers[:-8]:
    layer.trainable = False

print(f"Trainable layers: {sum([l.trainable for l in vgg_base.layers])} / {len(vgg_base.layers)}")

# recompile with a lower learning rate for fine tuning
vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# train on augmented data again
vgg_history_phase3 = vgg_model.fit(
    vgg_full_train_dataset,
    validation_data = vgg_val_dataset,
    epochs = VGG_EPOCHS,
    class_weight = class_weight,
    callbacks = [reduce_lr, early_stop]
)

# final evaluation
print("\n=== FINAL VGG16 EVALUATION ===")
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(vgg_test_dataset, verbose=0)
print(f"VGG16 final test accuracy: {vgg_test_acc:.4f}")


82/82 ━━━━━━━━━━━━━━━━━━━━ 46s 345ms/step - accuracy: 0.4175 - loss: 3.9430 - val_accuracy: 0.5430 - val_loss: 0.9514
Epoch 1/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 11s 135ms/step - accuracy: 0.5050 - loss: 1.4353 - val_accuracy: 0.5723 - val_loss: 0.9079 - learning_rate: 0.0010
Epoch 2/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 11s 134ms/step - accuracy: 0.5439 - loss: 1.1182 - val_accuracy: 0.5840 - val_loss: 0.8800 - learning_rate: 0.0010
Epoch 3/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 11s 132ms/step - accuracy: 0.6005 - loss: 0.9935 - val_accuracy: 0.6152 - val_loss: 0.8457 - learning_rate: 0.0010
Epoch 4/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 11s 133ms/step - accuracy: 0.6275 - loss: 0.8705 - val_accuracy: 0.6172 - val_loss: 0.8180 - learning_rate: 0.0010
Epoch 5/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 11s 132ms/step - accuracy: 0.6405 - loss: 0.8414 - val_accuracy: 0.6172 - val_loss: 0.7989 - learning_rate: 0.0010
Epoch 6/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 11s 133ms/step - accuracy: 0.6392 - loss: 0.8122 - val_accuracy: 0.6270 - val_l

In [ ]:
# save trained vgg model to google drive in case of colab disconnection
save_dir = "/content/drive/MyDrive/alz_models"
os.makedirs(save_dir, exist_ok=True)

vgg_path = os.path.join(save_dir, "vgg_best.keras")
vgg_model.save(vgg_path)

print("Saved VGG model to:", vgg_path)

Saved VGG model to: /content/drive/MyDrive/alz_models/vgg_best.keras


In [ ]:
# generate predictions and evaluation metrics
y_true = []
y_pred = []

for images, labels in vgg_test_dataset:
    preds = vgg_model.predict(images)

    true_classes = np.argmax(labels.numpy(), axis=1)

    y_true.extend(true_classes)
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 18s 7s/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
[[166   0   5  

In [4]:
# build EfficentNetB2 backbone
eff_base = EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(260, 260, 3),
)
eff_base.trainable = False

# EfficientNet classifier head
eff_inputs = tf.keras.Input(shape=(260, 260, 3))
y = eff_base(eff_inputs, training=False)
y = layers.GlobalAveragePooling2D()(y)
y = layers.Dense(256, activation="relu")(y)
y = layers.Dropout(0.5)(y)
eff_outputs = layers.Dense(4, activation="softmax")(y)

eff_model = tf.keras.Model(eff_inputs, eff_outputs)

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

eff_model.summary()

# use top_conv for gradcam

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 260, 260, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb2 (Functional)     │ (None, 9, 9, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       360,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,130,301 (31.01 MB)

 Trainable params: 361,732 (1.38 MB)

 Non-trainable params: 7,768,569 (29.63 MB)

In [ ]:
EFF_EPOCHS = 50

# same class weights as before with VGG
class_weight = {
    0: 1.25,
    1: 2.0,
    2: 1.0,
    3: 1.0,
}

#same reduce learning rate and early stopping as with VGG
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

# train for 1 epoch on clean original data
eff_history_phase0 = eff_model.fit(
    eff_clean_train,
    validation_data=eff_val_dataset,
    epochs=1,
    class_weight=class_weight,
)

# train briefly on clean & synthetic data
eff_history_phase1 = eff_model.fit(
    full_eff_synth_train,
    validation_data=eff_val_dataset,
    epochs=10,
    class_weight=class_weight,
    callbacks=[reduce_lr, early_stop]
)

# train on augmented data
eff_history_phase2 = eff_model.fit(
    eff_full_train_dataset,
    validation_data=eff_val_dataset,
    epochs=EFF_EPOCHS,
    class_weight=class_weight,
    callbacks = [reduce_lr, early_stop]
)

# unfreeze top layers of VGG16 to allow for fine tuning
print("\n=== PHASE 2: Fine-tuning top layers ===")
eff_base.trainable = True
for layer in eff_base.layers[:-8]:
    layer.trainable = False

print(f"Trainable layers: {sum([l.trainable for l in eff_base.layers])} / {len(eff_base.layers)}")

# recompile with a lower learning rate for fine tuning
eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# train on augmented data again
eff_history_phase3 = eff_model.fit(
    eff_full_train_dataset,
    validation_data=eff_val_dataset,
    epochs=EFF_EPOCHS,
    class_weight=class_weight,
    callbacks=[reduce_lr, early_stop],
)

# final evaluation
print("\n=== FINAL EfficientNetB2 EVALUATION ===")
eff_test_loss, eff_test_acc = eff_model.evaluate(eff_test_dataset, verbose=0)
print(f"EfficientNetB2 final test accuracy: {eff_test_acc:.4f}")

In [ ]:
# save trained efficientnet model to google drive in case of colab disconnection
save_dir = "/content/drive/MyDrive/alz_models"
os.makedirs(save_dir, exist_ok=True)

eff_path = os.path.join(save_dir, "eff_best.keras")
eff_model.save(eff_path)

print("Saved EfficientNet model to:", eff_path)

Saved EfficientNet model to: /content/drive/MyDrive/alz_models/eff_best.keras


In [ ]:
# evaluate EfficientNet predictions
y_true = []
y_pred = []

for images, labels in eff_test_dataset:
    preds = eff_model.predict(images)

    true_classes = np.argmax(labels.numpy(), axis=1)

    y_true.extend(true_classes)
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

In [ ]:
# get model probability outputs for the test set
vgg_probs = vgg_model.predict(vgg_test_dataset)
eff_probs = eff_model.predict(eff_test_dataset)

# average the two probability vectors for ensemble prediction
ensemble_probs = (vgg_probs + eff_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# recover true labels from the VGG dataset
onehot_labels = np.concatenate([y.numpy() for _, y in vgg_test_dataset], axis=0)
y_true = np.argmax(onehot_labels, axis=1)

# individual model predictions
vgg_preds = np.argmax(vgg_probs, axis=1)
eff_preds = np.argmax(eff_probs, axis=1)

print("VGG16 test accuracy: ", accuracy_score(y_true, vgg_preds))
print("EfficientNetB2 test acc: ", accuracy_score(y_true, eff_preds))
print("Ensemble test accuracy: ", accuracy_score(y_true, ensemble_preds))

print("\nEnsemble classification report:")
print(classification_report(y_true, ensemble_preds, digits=4))

print("\nEnsemble confusion matrix:")
print(confusion_matrix(y_true, ensemble_preds))

In [ ]:
# to use if Colab disconnects so I don't have to run the whole notebbok again
# load saved models back from drive
from tensorflow.keras.models import load_model

vgg_path = "/content/drive/MyDrive/alz_models/vgg_best.keras"
eff_path = "/content/drive/MyDrive/alz_models/eff_best.keras"

vggmodel = load_model(vgg_path, compile=False)
effmodel = load_model(eff_path, compile=False)

# predict on the test datasets using loaded models
vgg_probs = vggmodel.predict(vgg_test_dataset)
eff_probs = effmodel.predict(eff_test_dataset)

# average ensemble
ensemble_probs = (vgg_probs + eff_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# true labels
onehot_labels = np.concatenate([y.numpy() for _, y in vgg_test_dataset], axis=0)
y_true = np.argmax(onehot_labels, axis=1)

# individual model predictions
vgg_preds = np.argmax(vgg_probs, axis=1)
eff_preds = np.argmax(eff_probs, axis=1)

print("VGG16 test accuracy: ", accuracy_score(y_true, vgg_preds))
print("EfficientNetB2 test acc: ", accuracy_score(y_true, eff_preds))
print("Ensemble test accuracy: ", accuracy_score(y_true, ensemble_preds))

print("\nEnsemble classification report:")
print(classification_report(y_true, ensemble_preds, digits=4))

print("\nEnsemble confusion matrix:")
print(confusion_matrix(y_true, ensemble_preds))

26/26 ━━━━━━━━━━━━━━━━━━━━ 14s 279ms/step
26/26 ━━━━━━━━━━━━━━━━━━━━ 22s 469ms/step
VGG16 test accuracy:  0.9765625
EfficientNetB2 test acc:  0.89296875
Ensemble test accuracy:  0.97734375

Ensemble classification report:
              precision    recall  f1-score   support

           0     1.0000    0.9244    0.9607       172
           1     1.0000    0.9333    0.9655        15
           2     0.9828    0.9905    0.9866       634
           3     0.9615    0.9804    0.9709       459

    accuracy                         0.9773      1280
   macro avg     0.9861    0.9572    0.9709      1280
weighted avg     0.9777    0.9773    0.9773      1280


Ensemble confusion matrix:
[[159   0   2  11]
 [  0  14   0   1]
 [  0   0 628   6]
 [  0   0   9 450]]


In [ ]:
# export models that I save to disk to use in my application for single image classification
os.makedirs("export/models", exist_ok=True)

vgg_model.save("export/models/vgg_model.keras")
eff_model.save("export/models/eff_model.keras")
